# Model Initialization - Single `.mo` File

This notebook initializes one dynamic Modelica model from its auxiliary model.

It loads the model and its auxiliary from `models/`, simulates the auxiliary to compute the initialization values, applies them to a copy of the model, and saves the initialized model to `outputs/`.

In [ ]:
import logging
import os
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import pandas as pd
from OMPython import ModelicaSystemOMC, OMCSessionLocal

from scripts.helpers import *

# --- Configuration ---

# 1. Input and output directories
MODEL_DIR = os.path.abspath("models")
OUTPUT_DIR = os.path.abspath("outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Select the model to initialize
MODEL = "Demo_BESS"

# 3. Path to the selected dynamic case file
MODEL_FILE_PATH = os.path.join(MODEL_DIR, MODEL + ".mo")

# 4. Name and path of the initialized dynamic case file (written under outputs/)
INITIALIZED_MODEL = MODEL + "_initialized"
INITIALIZED_FILE_PATH = os.path.join(OUTPUT_DIR, INITIALIZED_MODEL + ".mo")

# 5. Path to the auxiliary file (input, from BuildAux)
AUX_MODEL = MODEL + "_auxiliary"
AUX_FILE_PATH = os.path.join(MODEL_DIR, AUX_MODEL + ".mo")

# 6. Modelica Standard Library
OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")

# 7. Dynawo Modelica library
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "dynawo_library", "Dynawo", "package.mo")
)

# 8. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = {}

# 9. Variable to plot after the initialized simulation
# PLOT_VARIABLE = "generatorSynchronous.terminal.i.re"  # SMIB option
PLOT_VARIABLE = "BESS.terminal.V.im"  # Demo_BESS option
# PLOT_VARIABLE = "PV.measurements.PPu" # Demo_PVCurrent option
# PLOT_VARIABLE = "deltaFrequency"  # Demo_DIGrid option

## Load and Validate the Models

The dynamic model and its auxiliary are loaded and checked, and the user configuration is validated against the dynamic model, before any changes are made.

In [ ]:
# 1. Load the dynamic model and validate the configuration against it
SourceOMC = OMCSessionLocal()
omc_call(SourceOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(SourceOMC, "loadModel(Complex)")
omc_call(SourceOMC, "loadModel(ModelicaServices)")
omc_call(SourceOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(SourceOMC, f'loadFile("{MODEL_FILE_PATH}")')
check_user_configuration_single(
    SourceOMC,
    model=MODEL,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)

# 2. Load the auxiliary model
AuxOMC = OMCSessionLocal()
omc_call(AuxOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(AuxOMC, "loadModel(Complex)")
omc_call(AuxOMC, "loadModel(ModelicaServices)")
omc_call(AuxOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(AuxOMC, f'loadFile("{AUX_FILE_PATH}")')
print("Checking the auxiliary model...")
chk_aux = send_expression(AuxOMC, f"checkModel({AUX_MODEL})", parsed=False)
print(chk_aux)

## Extract Initialization Values

The auxiliary model is simulated, and its outputs provide the initialization values to apply to the dynamic model.

In [ ]:
# 1. Build and simulate the auxiliary model
AuxSystem = ModelicaSystemOMC(session=AuxOMC)
AuxSystem.model(model_name=AUX_MODEL, model_file=AUX_FILE_PATH)
AuxSystem.simulate(resultfile=AUX_MODEL + "_res.mat")

# 2. Get the dynamic model components that need initialization
components = get_all_components(SourceOMC, MODEL)
initializable_components = get_initializable_components(components, INIT_MODEL_BY_COMPONENT)

# 3. Extract initialization values from the auxiliary model
init_values_by_component = extract_all_initialization_values(AuxSystem, components, INIT_MODEL_BY_COMPONENT)

## Build Initialized Model

A copy of the dynamic model is populated with the extracted values and saved as the initialized model in `outputs/`.

In [ ]:
send_expression(SourceOMC, f"deleteClass({INITIALIZED_MODEL})")
omc_call(SourceOMC, f'copyClass({MODEL}, "{INITIALIZED_MODEL}")')

apply_initialization_modifiers(
    SourceOMC, INITIALIZED_MODEL, initializable_components, init_values_by_component, INIT_MODEL_BY_COMPONENT
)

omc_call(SourceOMC, f'saveModel("{INITIALIZED_FILE_PATH}", {INITIALIZED_MODEL})')
print("Wrote initialized model:", INITIALIZED_FILE_PATH)

## Validate Initialized Model

The initialized model is loaded independently and simulated to verify that the final output can run.

In [ ]:
InitializedOMC = OMCSessionLocal()
omc_call(InitializedOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(InitializedOMC, "loadModel(Complex)")
omc_call(InitializedOMC, "loadModel(ModelicaServices)")
omc_call(InitializedOMC, f'loadFile("{DYNAWO_PKG_PATH}")')

InitializedSystem = ModelicaSystemOMC(session=InitializedOMC)
InitializedSystem.model(model_name=INITIALIZED_MODEL, model_file=INITIALIZED_FILE_PATH)
print("Checking the initialized model...")
chk_init = send_expression(InitializedOMC, f"checkModel({INITIALIZED_MODEL})", parsed=False)
print(chk_init)

simflags = simulation_flags_without_log_stats(InitializedOMC, INITIALIZED_MODEL)
initialized_resultfile_prefix = INITIALIZED_MODEL

sim_result = send_expression(
    InitializedOMC,
    f'simulate({INITIALIZED_MODEL}, outputFormat="csv", '
    f'fileNamePrefix="{initialized_resultfile_prefix}", simflags="{simflags}")',
    parsed=False,
)

print(sim_result)

initialized_resultfile = os.path.join(
    str(InitializedSystem.getWorkDirectory()),
    initialized_resultfile_prefix + "_res.csv",
)

## Optional: Plot Simulation Result

The final cell is an optional example of visualizing a variable from the initialized-model simulation result.

In [ ]:
initialized_df = pd.read_csv(initialized_resultfile)
if PLOT_VARIABLE not in initialized_df.columns:
    raise RuntimeError(f'PLOT_VARIABLE "{PLOT_VARIABLE}" is not a variable in the simulation result.')

fig, ax = plt.subplots()
ax.plot(initialized_df["time"], initialized_df[PLOT_VARIABLE], label=PLOT_VARIABLE)
ax.legend(loc="lower right")
ax.set_title("Initialized dynamic model response")
ax.set_xlabel("Time (s)")
ax.set_ylabel(PLOT_VARIABLE)